# LLM 4일차 (1) — 벡터 저장소 · 검색 도구 · RAG 체인 : 지능형 고객 응대 Chatbot 완성

수업 필기 `RAG_실습07` 을 정리 · 개선한 노트입니다. 어제(Day 048)는 준비 단계의 ① 불러오기 ② 분할까지, 오늘은 ③ 임베딩 ④ 벡터 저장소 · 검색 도구 ⑤ 프롬프트 · LLM ⑥ RAG 체인 ⑦ 챗봇 실행까지.
(교재 `AI Agent` **p.39~43 · 45~51** — 실습 과정 7단계 · 벡터 데이터베이스 · FAISS vs ChromaDB · SQLite · 메타데이터 필터링)

```python
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory='chroma_db')  # ③ ④ 임베딩 + 저장
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})                                               # ④ 검색 도구
rag_chain = prompt | llm | StrOutputParser()                                                               # ⑤ ⑥
rag_chain.invoke({'context': 검색된_글, 'question': 질문})                                                  # ⑦
```

- PDF 헌법 Q&A 챗봇(필기 `RAG_실습08` · 강의본 `RAG_실습8_1`)은 [`rag_pdf_constitution.ipynb`](./rag_pdf_constitution.ipynb)
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

> **출력 읽는 법** — 정리 노트북은 **API 키 없이** 만들었다(Day 046 ~ 048 과 같은 방식).
>
> | 표시 | 뜻 |
> |---|---|
> | `# 📼` | **수업 시간에 실제로 실행한 결과**를 필기 원본에서 그대로 옮긴 출력 |
> | `# ⏳` | API 키가 필요해 **출력이 비어 있는 셀** — 직접 실행 |
> | `# 🧪` | OpenAI 대신 **로컬 임베딩 모델**(`jhgan/ko-sroberta-multitask`, Day 041 · 045 에서 쓴 것) 또는 **가짜 LLM** 으로 실제로 돌린 셀 — 검색 순위 · 점수는 수업의 `text-embedding-3-small` 과 다를 수 있다 |
> | 표시 없음 | 정리 노트북에서 실제로 실행한 셀 (불러오기 · 분할 · DB 파일 들여다보기 · 수업 기록 분석) |

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `langchain-community` · `langchain-text-splitters` | `TextLoader` · `RecursiveCharacterTextSplitter` · `FAISS` | `pip install langchain-community langchain-text-splitters` |
| `langchain-chroma` · `chromadb` | `Chroma` 벡터 저장소 (수업의 `langchain_community.vectorstores.Chroma` 는 지원 종료 예정) | `pip install langchain-chroma` |
| `faiss-cpu` | `FAISS` 비교 | `pip install faiss-cpu` |
| `langchain-openai` | `OpenAIEmbeddings` · `ChatOpenAI` (⏳ 셀) | `pip install langchain-openai` |
| `langchain-huggingface` · `sentence-transformers` | 🧪 로컬 한국어 임베딩 (키 없이 검색 실습) | `pip install langchain-huggingface sentence-transformers` |
| `python-dotenv` · `pandas` | API 키 · 표 | `pip install python-dotenv pandas` |

**데이터** : 노트북과 같은 폴더의 `manual.txt` (Day 048 과 같은 파일). **API 키** : `.env` → `OPENAI_API_KEY="sk-…"` (**GitHub 에 올리지 않는다**).
**실행** : 맨 위부터 **모두 실행**. 🧪 로컬 임베딩은 처음 한 번 Hugging Face 에서 모델(약 440MB)을 받으므로 인터넷이 필요하고, 그다음부터는 키 · 인터넷 없이 돈다. 노트북 폴더에 연습용 DB 폴더(`chroma_db_*` · `faiss_manual`)가 생긴다.

In [1]:
# ① 설치 (처음 한 번)
# !pip install -q langchain-community langchain-text-splitters langchain-chroma faiss-cpu langchain-openai langchain-huggingface sentence-transformers python-dotenv pandas

In [2]:
import os, re, shutil, sqlite3, warnings
import pandas as pd
warnings.filterwarnings('ignore', message='.*sunset.*')              # langchain-community 지원 종료 예고 경고 숨기기
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma                                  # 📝 langchain_community.vectorstores.Chroma 대신
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 이 노트북이 만드는 연습용 DB 폴더는 매번 새로 만든다 (다시 실행해도 같은 결과가 나오게)
for d in ['chroma_db_demo', 'chroma_db_manual_160', 'chroma_db_manual_110', 'chroma_db_manual_cos', 'chroma_db_manual_meta', 'faiss_manual']:
    shutil.rmtree(d, ignore_errors=True)

In [3]:
# .env 파일 → 환경 변수 (키 값은 절대 print 하지 않는다)
from dotenv import load_dotenv, find_dotenv
loaded = load_dotenv(find_dotenv(usecwd=True))      # 지금 폴더에서 위로 올라가며 .env 를 찾는다
print('.env 읽음 :', loaded, '| OPENAI_API_KEY 있음 :', bool(os.getenv('OPENAI_API_KEY')))

.env 읽음 : False | OPENAI_API_KEY 있음 : False


## 1. 실습 과정 7단계 (교재 p.39)

| 단계 | 하는 일 | 코드 | 언제 |
|---|---|---|---|
| ① 데이터 수집 · 변환 | 텍스트 → `Document` | `TextLoader(...).load()` | 준비 (한 번) — Day 048 |
| ② 텍스트 분할 | 긴 문서 → 청크 | `RecursiveCharacterTextSplitter(...).split_documents()` | 준비 — Day 048 |
| ③ 텍스트 임베딩 | 청크 → 숫자 벡터 | `OpenAIEmbeddings(model='text-embedding-3-small')` | 준비 |
| ④ 벡터 저장소 · 검색 도구 | 벡터 + 원문 저장 · 비슷한 청크 찾기 | `Chroma.from_documents(...)` · `.as_retriever(k=…)` | 준비 / 질문마다 검색 |
| ⑤ 프롬프트 · LLM | `{context}` · `{question}` 자리 | `ChatPromptTemplate` · `ChatOpenAI` | 질문마다 |
| ⑥ RAG 체인 | 프롬프트 \| LLM \| 파서 | `prompt \| llm \| StrOutputParser()` | 질문마다 |
| ⑦ 챗봇 실행 | 질문 → 검색 → 증강 → 생성 반복 | `while` / `ask()` 함수 | 질문마다 |

## 2. ① 불러오기 · ② 분할 — 수업 값 `chunk_size=110` 다시 보기

In [4]:
documents = TextLoader('manual.txt', encoding='utf-8').load()
manual_text = documents[0].page_content
print('Document', len(documents), '개 ·', len(manual_text), '자 · QA 쌍', len(manual_text.split('\n\n')), '개')

def split(chunk_size):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0).split_documents(documents)

def kind(text):                         # 청크 하나가 어떤 조각인지
    if text.startswith('질문') and '답변' in text: return 'QA 한 쌍'
    if text.startswith('질문'): return '질문만'
    if text.startswith('답변'): return '답변 (끝까지)' if text.rstrip().endswith(('.', '요')) else '답변 (중간에 잘림)'
    return '잘린 꼬리 조각'

chunks110, chunks160 = split(110), split(160)
pd.DataFrame({'chunk_size=110 (수업)': pd.Series([kind(c.page_content) for c in chunks110]).value_counts(),
              'chunk_size=160 (Day 048)': pd.Series([kind(c.page_content) for c in chunks160]).value_counts()}).fillna(0).astype(int)

Document 1 개 · 1399 자 · QA 쌍 10 개


,chunk_size=110 (수업),chunk_size=160 (Day 048)
QA 한 쌍,0,10
답변 (끝까지),5,0
답변 (중간에 잘림),5,0
잘린 꼬리 조각,5,0
질문만,10,0


In [5]:
print('110 에서 생긴 꼬리 조각 :', [c.page_content for c in chunks110 if kind(c.page_content) == '잘린 꼬리 조각'])

110 에서 생긴 꼬리 조각 : ['수 있습니다.', '즉시 기술 지원팀에 문의해 주세요.', '수도 있습니다.', '드립니다.', '이용하실 수 있습니다.']


> **📝 고친 부분 (원본 cell 4 · `chunk_size=110`)** 수업 값 110 은 가장 긴 QA 쌍(157자)보다 작아서 **청크 25개** — 질문 10개는 **답 없이 질문만**, 답변 5개는 끝이 잘려 `드립니다.` · `수 있습니다.` 같은 **꼬리 조각** 5개가 따로 생겼다(수업 출력의 25개와 같다). Day 048 에서 잰 대로 QA 한 쌍 = 청크 하나가 되려면 `chunk_size` 는 **159 ~ 238** → 이 노트는 **160** 을 쓴다. 이 차이가 검색 결과에 어떻게 나오는지는 §5.

## 3. ③ 텍스트 임베딩

In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector = embeddings.embed_query('환불 받고 싶어요')         # 질문 1개 → 벡터
print(len(vector), vector[:5])                               # 1536 차원
vectors = embeddings.embed_documents([c.page_content for c in chunks160])   # 청크 여러 개 → 벡터 여러 개
print(len(vectors), len(vectors[0]))

In [7]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

from langchain_huggingface import HuggingFaceEmbeddings
local_emb = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask',      # 처음 한 번 약 440MB 내려받음
                                  encode_kwargs={'normalize_embeddings': True})  # 길이 1 로 → 내적 = 코사인
v = local_emb.embed_query('환불 받고 싶어요')
print('벡터 길이 :', len(v), '| 한 번에 읽는 최대 토큰 :', local_emb._client.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터 길이 : 768 | 한 번에 읽는 최대 토큰 : 128


- 임베딩 모델은 `embed_documents`(저장할 청크들) · `embed_query`(질문) 두 함수만 있으면 된다 — 그래서 `OpenAIEmbeddings` 자리에 로컬 모델을 그대로 끼울 수 있다. **저장할 때와 검색할 때 같은 모델**을 써야 한다(벡터 길이 · 공간이 다르면 비교가 안 된다)
- `text-embedding-3-small` 은 1,536 차원 · 입력 최대 8,191 토큰, 로컬 `ko-sroberta-multitask` 는 768 차원 · **128 토큰까지만** 읽는다(이 매뉴얼의 청크는 모두 그 안)

## 4. ④ 벡터 저장소 — `Chroma` (교재 p.40 · 47~50)

In [8]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

chroma_path = 'chroma_db_manual_160'
vectorstore = Chroma.from_documents(documents=chunks160, embedding=local_emb, persist_directory=chroma_path)
print('저장된 청크 수 :', vectorstore._collection.count())
stored = vectorstore.get(limit=2, include=['documents', 'metadatas', 'embeddings'])
for i in range(2):
    print(stored['ids'][i][:8], stored['metadatas'][i], stored['documents'][i][:30].replace('\n', ' '), '| 벡터 앞 3칸', stored['embeddings'][i][:3].round(4))

저장된 청크 수 : 10
a1d197dc {'source': 'manual.txt'} 질문: 코드크래프터스 아카데미에 어떻게 가입하나요? 답 | 벡터 앞 3칸 [-0.0048  0.009   0.0332]
64a074c3 {'source': 'manual.txt'} 질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책 | 벡터 앞 3칸 [-0.0061  0.0191  0.0193]


In [9]:
# persist_directory 안에 실제로 무엇이 생겼나
for root, dirs, files in os.walk(chroma_path):
    for f in files:
        print(f'{os.path.join(root, f):75s} {os.path.getsize(os.path.join(root, f)):>9,} bytes')

chroma_db_manual_160/chroma.sqlite3                                           249,856 bytes
chroma_db_manual_160/74f541fa-da55-47f3-84f7-350447e907a9/length.bin              400 bytes
chroma_db_manual_160/74f541fa-da55-47f3-84f7-350447e907a9/link_lists.bin            0 bytes
chroma_db_manual_160/74f541fa-da55-47f3-84f7-350447e907a9/data_level0.bin     321,200 bytes
chroma_db_manual_160/74f541fa-da55-47f3-84f7-350447e907a9/header.bin              100 bytes


In [10]:
# chroma.sqlite3 를 직접 열어 보기 (교재 p.50 의 embedding_metadata · embeddings_queue)
con = sqlite3.connect(f'{chroma_path}/chroma.sqlite3')
tables = [r[0] for r in con.execute("select name from sqlite_master where type='table'")]
print('테이블', len(tables), '개 :', [t for t in tables if t.startswith(('embedding', 'collections', 'segments'))])
print('embedding_metadata 의 key 종류 :', con.execute('select distinct key from embedding_metadata').fetchall())
print('원문 예 :', con.execute("select substr(string_value, 1, 30) from embedding_metadata where key='chroma:document' limit 1").fetchone())
row = con.execute('select operation, length(vector), encoding from embeddings_queue limit 1').fetchone()
print(f'embeddings_queue 한 줄 : 벡터 {row[1]:,} bytes = 768 차원 × 4 bytes ({row[2]}) · 줄 수', con.execute('select count(*) from embeddings_queue').fetchone()[0])
con.close()

테이블 21 개 : ['collections', 'segments', 'embeddings', 'embedding_metadata', 'embedding_fulltext_search', 'embedding_fulltext_search_data', 'embedding_fulltext_search_idx', 'embedding_fulltext_search_content', 'embedding_fulltext_search_docsize', 'embedding_fulltext_search_config', 'embedding_metadata_array', 'embeddings_queue', 'embeddings_queue_config']
embedding_metadata 의 key 종류 : [('chroma:document',), ('source',)]
원문 예 : ('질문: 코드크래프터스 아카데미에 어떻게 가입하나요?\n답',)
embeddings_queue 한 줄 : 벡터 3,072 bytes = 768 차원 × 4 bytes (FLOAT32) · 줄 수 10


> **📝 고친 부분 (교재 p.50 · 원본 cell 5 주석)** '원본 텍스트 : `embedding_metadata` · 임베딩 벡터 : `embeddings_queue` 테이블에 저장' →
> - `embedding_metadata` 에는 원문(`chroma:document`)과 **메타데이터**(`source` …)가 함께 들어 있다 — 맞다
> - `embeddings_queue` 는 이름 그대로 **쓰기 대기열(기록 로그)** 이다. 벡터가 들어 있기는 하지만, 검색에 실제로 쓰는 벡터 **인덱스는 옆의 UUID 폴더**(`data_level0.bin` 등, HNSW 그래프)에 있다. 원본 필기 출력의 `'c23a45ad-…'` 폴더가 바로 그것
> - 원본 주석 '`from_documents()` : 임베딩하고 **메모리에** 저장' → `persist_directory` 를 주면 **디스크**에 저장된다(주지 않으면 메모리)

## 5. 📝 '폴더가 있으면 불러오기' 의 함정 — 설정을 바꿔도 옛날 DB

In [11]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

def load_or_create(chunks, path='chroma_db_demo'):          # 수업 코드와 같은 분기
    if os.path.exists(path) and os.listdir(path):
        print('기존 DB 를 불러옴 →', end=' ')
        vs = Chroma(persist_directory=path, embedding_function=local_emb)
    else:
        print('새 DB 를 만듦     →', end=' ')
        vs = Chroma.from_documents(documents=chunks, embedding=local_emb, persist_directory=path)
    print('청크', vs._collection.count(), '개')
    return vs

first = load_or_create(chunks110)        # 처음 : 110 으로 잘라 25개 저장
again = load_or_create(chunks160)        # chunk_size 를 160 으로 고쳐 다시 실행 → 폴더가 있으니 그대로 25개!

새 DB 를 만듦     → 

청크 25 개
기존 DB 를 불러옴 → 청크 25 개


- 수업 코드는 폴더가 있으면 **무조건 불러온다.** 그래서 `chunk_size` · 데이터 · 임베딩 모델을 바꾼 뒤 다시 실행해도 **처음 만든 DB 가 그대로** 쓰인다 — 원본 필기 출력 `경로2 : ['c23a…', 'chroma.sqlite3']` · `__is not new___` 가 이 상태다
- 해결 : ① 폴더 이름에 설정을 넣는다(`chroma_db_manual_160`) ② 설정을 바꿨으면 폴더를 지우고 **커널을 다시 시작한 뒤** 만든다(같은 프로세스에서 쓰던 DB 폴더를 지우고 다시 만들면 `attempt to write a readonly database` 오류) ③ 불러온 뒤 `vs._collection.count()` 가 기대한 청크 수인지 확인

## 6. ④ 검색 도구 — `chunk_size` 110 vs 160 (교재 p.42)

In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

retriever = vectorstore.as_retriever(search_kwargs={'k':1})
user_query = "환불 해줘"
output = retriever.invoke(user_query)
print("검색 결과:",output)

검색 결과: [Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.')]


In [13]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

store110 = Chroma.from_documents(chunks110, local_emb, persist_directory='chroma_db_manual_110')
rows = []
for q in ['환불 해줘', '돈 돌려줘', '수강 신청한 강의를 환불하고 싶어요', '핸드폰으로도 들을 수 있어?', '강사님께 질문하려면?']:
    for name, store in [('110', store110), ('160', vectorstore)]:
        top = store.as_retriever(search_kwargs={'k': 1}).invoke(q)[0].page_content
        rows.append({'질문': q, 'chunk_size': name, '1등 청크': top[:38].replace('\n', ' '), '종류': kind(top), '답 들어 있음': '답변' in top})
pd.DataFrame(rows)

,질문,chunk_size,1등 청크,종류,답 들어 있음
0,환불 해줘,110,드립니다.,잘린 꼬리 조각,False
1,환불 해줘,160,질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나,QA 한 쌍,True
2,돈 돌려줘,110,드립니다.,잘린 꼬리 조각,False
3,돈 돌려줘,160,질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나,QA 한 쌍,True
4,수강 신청한 강의를 환불하고 싶어요,110,질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나,질문만,False
5,수강 신청한 강의를 환불하고 싶어요,160,질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나,QA 한 쌍,True
6,핸드폰으로도 들을 수 있어?,110,질문: 모바일 앱으로도 강의를 들을 수 있나요?,질문만,False
7,핸드폰으로도 들을 수 있어?,160,"질문: 모바일 앱으로도 강의를 들을 수 있나요? 답변: 네, 가능합니",QA 한 쌍,True
8,강사님께 질문하려면?,110,질문: 강의 자료나 소스 코드는 어디서 다운로드받나요?,질문만,False
9,강사님께 질문하려면?,160,질문: 코딩 실습 중에 모르는 것이 생기면 어떻게 하죠? 답변: 각,QA 한 쌍,True


- `chunk_size=110` 이면 1등이 **질문 줄만**이거나 `드립니다.` 같은 **꼬리 조각**이다 — LLM 이 받는 컨텍스트에 답이 없다. 수업 챗봇 실행(§10 📼)에서도 검색된 것은 '질문: 수강 신청한 강의를 환불하고 싶어요…' **질문 줄 하나**였다
- 질문과 똑같은 질문 줄이 가장 비슷한 것은 당연하다 — 그래서 질문과 답을 **한 청크에 묶어야** 질문으로 찾고 답을 얻는다
- `k=1` 은 하나만 가져오므로 실수에 약하다. 수업 PDF 실습처럼 `k=2` 이상이 안전하다

## 7. 점수와 '매뉴얼에 없는 질문'

In [14]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

# 거리 계산을 코사인으로 지정하면 relevance score = 코사인 유사도 (Chroma 기본은 l2)
cos_store = Chroma.from_documents(chunks160, local_emb, persist_directory='chroma_db_manual_cos',
                                  collection_configuration={'hnsw': {'space': 'cosine'}})
print('기본 거리 :', vectorstore._collection.configuration['hnsw']['space'], '→', cos_store._collection.configuration['hnsw']['space'])
for q in ['환불 해줘', '돈 돌려줘', '수료증 나와요?', '핸드폰으로도 들을 수 있어?', '회사 주소가 어디예요?', '주차는 가능한가요?']:
    (d1, s1), (d2, s2) = cos_store.similarity_search_with_relevance_scores(q, k=2)
    print(f'{q:18s} 1등 {s1:.3f} {d1.page_content[4:24]:22s} | 2등 {s2:.3f}')

기본 거리 : l2 → cosine
환불 해줘              1등 0.493 수강 신청한 강의를 환불하고 싶어요.   | 2등 0.302
돈 돌려줘              1등 0.404 수강 신청한 강의를 환불하고 싶어요.   | 2등 0.339
수료증 나와요?           1등 0.592 강의를 다 들으면 수료증을 받을 수    | 2등 0.386
핸드폰으로도 들을 수 있어?    1등 0.458 모바일 앱으로도 강의를 들을 수 있나   | 2등 0.328


회사 주소가 어디예요?       1등 0.338 코드크래프터스 아카데미에 어떻게 가입   | 2등 0.304
주차는 가능한가요?         1등 0.279 어떤 결제 수단을 사용할 수 있나요?   | 2등 0.272


- 검색 도구는 **항상 k개를 돌려준다** — 매뉴얼에 없는 '회사 주소' · '주차' 도 가장 덜 먼 청크를 가져온다. 이것을 걸러 주는 것이 프롬프트 규칙 3('정보가 없다면 … 아는 바가 없습니다')이다
- 점수로 미리 거를 수도 있다 : `as_retriever(search_type='similarity_score_threshold', search_kwargs={'score_threshold': 0.4})`. 다만 기준값은 **모델 · 데이터마다 다르다** — 이 모델에서는 뜻이 같은 질문도 0.40 ~ 0.6 이라 여유가 크지 않다(Day 045 챗봇 기준값과 같은 이야기)
- Chroma 기본 거리 `l2` 에서는 LangChain 점수가 코사인과 다른 숫자(음수도 가능)가 된다. 순서는 같다(길이 1 벡터라서)

## 8. FAISS vs ChromaDB (교재 p.45~49)

In [15]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(chunks160, local_emb)            # 교재 p.49 : 저장소 이름만 바꾸면 된다
q = '강의 동영상이 안 나와요'
print('FAISS  :', [d.page_content[4:20] for d in faiss_store.similarity_search(q, k=3)])
print('Chroma :', [d.page_content[4:20] for d in vectorstore.similarity_search(q, k=3)])

faiss_store.save_local('faiss_manual')                              # FAISS 는 저장을 직접 해야 한다
print('저장된 파일 :', sorted(os.listdir('faiss_manual')))
try:
    FAISS.load_local('faiss_manual', local_emb)
except ValueError as e:
    print('그냥 불러오면 :', str(e)[:75], '…')
loaded = FAISS.load_local('faiss_manual', local_emb, allow_dangerous_deserialization=True)   # 내가 만든 파일일 때만
print('다시 불러온 벡터 수 :', loaded.index.ntotal)

FAISS  : ['강의 동영상이 자꾸 끊기거나 ', '선수 지식이 없는데, 어떤 강', '강의 자료나 소스 코드는 어디']
Chroma : ['강의 동영상이 자꾸 끊기거나 ', '선수 지식이 없는데, 어떤 강', '강의 자료나 소스 코드는 어디']
저장된 파일 : ['index.faiss', 'index.pkl']
그냥 불러오면 : The de-serialization relies loading a pickle file. Pickle files can be modi …
다시 불러온 벡터 수 : 10


| | FAISS | ChromaDB |
|---|---|---|
| 만들기 | `FAISS.from_documents(docs, emb)` | `Chroma.from_documents(docs, emb, persist_directory=…)` |
| 저장 | **직접** `save_local(폴더)` → `index.faiss` + `index.pkl` | `persist_directory` 를 주면 **자동** (`chroma.sqlite3` + 인덱스 폴더) |
| 불러오기 | `FAISS.load_local(폴더, emb, allow_dangerous_deserialization=True)` — `index.pkl` 은 피클이라 **남이 준 파일은 열지 않는다** | `Chroma(persist_directory=…, embedding_function=emb)` |
| 메타데이터 필터 | `filter={'category': '결제'}` 정도는 된다(검색 **후** 거름, `fetch_k` 개 안에서) | `where` 처럼 `$eq` · `$and` · `$in` … (검색 **전** 거름) |

- 교재 p.49 '코드 1줄 변경만으로 교체' → **만드는 줄**은 1줄이지만, **저장 · 불러오기** 코드는 위처럼 다르다
- 교재 p.46 'FAISS 는 메타데이터 검색이 제한적' → LangChain 의 FAISS 도 간단한 필터는 된다. 다만 먼저 `fetch_k`(기본 20)개를 찾은 뒤 거르므로, 조건에 맞는 문서가 그 안에 없으면 덜 나온다

## 9. 메타데이터 필터링 (교재 p.51)

In [16]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다

categories = ['계정', '결제', '수강', '결제', '기술지원', '수강', '수강', '학습지원', '학습지원', '기술지원']   # QA 10쌍에 직접 붙인 분류
for i, c in enumerate(chunks160):
    c.metadata.update(qa_id=i + 1, category=categories[i])
meta_store = Chroma.from_documents(chunks160, local_emb, persist_directory='chroma_db_manual_meta',
                                   collection_configuration={'hnsw': {'space': 'cosine'}})

q = '어떻게 해야 하나요?'                                         # 무엇을 묻는지 모호한 질문
show = lambda docs: [(d.metadata['qa_id'], d.metadata['category'], d.page_content[4:22]) for d in docs]
print('필터 없음          :', show(meta_store.as_retriever(search_kwargs={'k': 2}).invoke(q)))
print("category = '결제'  :", show(meta_store.as_retriever(search_kwargs={'k': 2, 'filter': {'category': '결제'}}).invoke(q)))
print('수강 and qa_id ≥ 6 :', show(meta_store.as_retriever(search_kwargs={'k': 2, 'filter': {
    '$and': [{'category': {'$eq': '수강'}}, {'qa_id': {'$gte': 6}}]}}).invoke('자료')))
print("$in 기술지원·계정  :", show(meta_store.as_retriever(search_kwargs={'k': 2, 'filter': {
    'category': {'$in': ['기술지원', '계정']}}}).invoke(q)))

필터 없음          : [(9, '학습지원', '선수 지식이 없는데, 어떤 강의부'), (4, '결제', '어떤 결제 수단을 사용할 수 있나')]
category = '결제'  : [(4, '결제', '어떤 결제 수단을 사용할 수 있나'), (2, '결제', '수강 신청한 강의를 환불하고 싶어')]
수강 and qa_id ≥ 6 : [(6, '수강', '강의 자료나 소스 코드는 어디서 '), (7, '수강', '수강 기간에 제한이 있나요?\n답변')]
$in 기술지원·계정  : [(1, '계정', '코드크래프터스 아카데미에 어떻게 '), (5, '기술지원', '강의 동영상이 자꾸 끊기거나 재생')]


- 교재 p.51 의 `{'$and': [{'category': {'$eq': '보안'}}, {'year': {'$eq': 2026}}]}` 와 같은 모양 — SQL 의 `where category='보안' and year=2026`
- 필터는 **먼저 후보를 줄이고**, 그 안에서 유사도 순위를 매긴다. 조건이 둘 이상이면 `$and` · `$or` 로 감싸야 한다(딕셔너리에 키 두 개를 그냥 넣으면 Chroma 가 오류)
- 메타데이터는 **분할할 때 붙여야** 한다. 매뉴얼에는 분류가 없어서 직접 붙였다 — PDF 헌법은 '장(章)' 을 붙일 수 있다([`rag_pdf_constitution.ipynb`](./rag_pdf_constitution.ipynb))

## 10. ⑤ ⑥ ⑦ RAG 체인 · 챗봇 (교재 p.43)

In [17]:
system_prompt = '''
당신은 친절한 온라인 교육 플랫폼 직원입니다.
제공된 [컨텍스트]를 바탕으로 고객의 질문에 정중하게 답변해주세요.

[답변 규칙]
1. [컨텍스트]의 내용을 바탕으로 자연스럽게 답변을 재구성하거나 그대로 활용하세요.
2. 답변 시 '답변:'이라는 표현을 사용하지 마세요.
3. [컨텍스트]에 질문에 대한 정보가 없다면, '죄송하지만 문의하신 내용에 대해서는 아는 바가 없습니다.'라고 답변하세요.

[컨텍스트]
{context}
'''
prompt = ChatPromptTemplate.from_messages([('system', system_prompt), ('user', '{question}')])
print('채울 변수 :', prompt.input_variables)

def format_docs(docs):                       # 검색된 Document 여러 개 → 글 하나
    return '\n\n'.join(d.page_content for d in docs)

채울 변수 : ['context', 'question']


In [ ]:
# 📼 아래 출력 = 수업 시간에 실행한 결과 (필기 원본에서 옮김)

print("프로그램 실행합니다.!/종료 (q)")
print("="*50)
while True:
    try:
        user_query = input("Q&A:")
        retriever_docs = retriever.invoke(user_query)
        if user_query.lower() == "q":
            print("프로그램 종료")
            break
        else:
            print("->",retriever_docs)
            print("--log--")
            context_text = retriever_docs[0].page_content
            print(context_text)
        print("=="*50)
        response = rag_chain.invoke({'conttext':context_text, 'question':user_query})   # 수업 때 실행된 코드 (오타)
    except Exception as e:
        print("ERRROR!",e)
        break

프로그램 실행합니다.!/종료 (q)
-> [Document(metadata={'source': 'C:\\Users\\Jinho\\work\\manual.txt'}, page_content='질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?')]
--log--
질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
ERRROR! "Input to ChatPromptTemplate is missing variables {'context'}.  Expected: ['context', 'question'] Received: ['conttext', 'question']\nNote: if you intended {context} to be part of the string and not a variable, please escape it with double curly braces like: '{{context}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "


> **📝 고친 부분 (원본 cell 9)**
> - 수업 출력의 오류 `missing variables {'context'} … Received: ['conttext', 'question']` — 넘긴 키 `conttext` 오타. 필기 코드는 `'context'` 로 고쳐져 있지만 **다시 실행하지 않아** 오류 출력이 남아 있다
> - 고쳐도 `response` 를 **출력하지 않는다** → 답이 화면에 안 나온다 → `print(response)`
> - `retriever.invoke(user_query)` 가 종료 확인(`q`)보다 **먼저** 실행 → `q` 를 입력해도 검색(임베딩 API 호출)을 한 번 한다 → 종료 확인을 맨 앞으로
> - 검색된 것은 질문 줄 하나뿐 — `chunk_size=110` 때문(§6). 160 으로 고치면 QA 한 쌍이 통째로 온다
> - `retriever_docs[0].page_content` 는 `k` 를 늘려도 첫 문서만 쓴다 → `format_docs(docs)` 로 모두 이어 붙인다
> - 원본 cell 6 의 주석 처리 코드 `vectorstore.index_to_docstore_id` · `get_by_ids` 는 **FAISS** 용 → Chroma 는 `vectorstore.get()`. 맨 끝의 `""" … """` 문자열이 셀의 마지막 값이라 **주석인데 출력**으로 찍혔다

In [19]:
# 🧪 LLM 대신 '받은 컨텍스트의 첫 줄을 되돌려 주는' 가짜 LLM — 답 내용이 아니라 체인에 무엇이 들어가는지 확인용

from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

def _fake_llm(prompt_value):
    system = prompt_value.to_messages()[0].content
    context = system.split('[컨텍스트]')[-1].strip()
    first = context.splitlines()[0] if context else '(컨텍스트 없음)'
    return AIMessage(f'(모의 답변 · 컨텍스트 {len(context)}자를 받음) {first[:70]}')

fake_llm = RunnableLambda(_fake_llm)

In [20]:
# 🧪 OpenAI 임베딩 대신 로컬 한국어 임베딩(jhgan/ko-sroberta-multitask)으로 실제 실행 — 순위 · 점수는 text-embedding-3-small 과 다를 수 있다
#   + LLM 대신 '받은 컨텍스트의 첫 줄을 되돌려 주는' 가짜 LLM — 답 내용이 아니라 체인에 무엇이 들어가는지 확인용

retriever = cos_store.as_retriever(search_kwargs={'k': 2})
fake_chain = prompt | fake_llm | StrOutputParser()

def ask(question, chain=fake_chain, retriever=retriever, show_context=True):
    docs = retriever.invoke(question)                     # ④ 검색
    context = format_docs(docs)                           # 증강할 글
    if show_context:
        print('  검색 :', [d.page_content[4:26].replace('\n', ' ') for d in docs])
    return chain.invoke({'context': context, 'question': question})   # ⑥ 증강 → 생성

for q in ['환불 해줘', '수료증 나와요?']:
    print('질문 :', q); print('  답  :', ask(q))

질문 : 환불 해줘
  검색 : ['수강 신청한 강의를 환불하고 싶어요. 환', '강의 동영상이 자꾸 끊기거나 재생되지 않']
  답  : (모의 답변 · 컨텍스트 301자를 받음) 질문: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?
질문 : 수료증 나와요?


  검색 : ['강의를 다 들으면 수료증을 받을 수 있나', '코드크래프터스 아카데미에 어떻게 가입하나']
  답  : (모의 답변 · 컨텍스트 281자를 받음) 질문: 강의를 다 들으면 수료증을 받을 수 있나요?


In [21]:
# LLM 이 실제로 받는 메시지 — 시스템 메시지 안의 [컨텍스트] 자리에 검색된 청크 2개가 들어갔다 (= 증강)
docs = retriever.invoke('환불 해줘')
for m in prompt.format_messages(context=format_docs(docs), question='환불 해줘'):
    print(f'── {type(m).__name__} ({len(m.content)}자)'); print(m.content.strip()[-260:])

── SystemMessage (533자)
답변: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.

질문: 강의 동영상이 자꾸 끊기거나 재생되지 않아요.
답변: 동영상 재생에 불편을 드려 죄송합니다. 우선 사용하고 계신 인터넷 브라우저의 캐시를 삭제해 보시거나, 다른 브라우저(크롬, 엣지 등)에서 재생해 보시길 권장합니다. 문제가 지속될 경우, 즉시 기술 지원팀에 문의해 주세요.
── HumanMessage (5자)
환불 해줘


In [ ]:
# ⏳ API 키가 필요한 셀 — 정리 환경에서는 실행하지 않음 (직접 실행해 보기)

# 진짜 RAG : OpenAI 임베딩으로 DB 를 만들고 ChatOpenAI 로 답 생성
openai_store = Chroma.from_documents(chunks160, OpenAIEmbeddings(model='text-embedding-3-small'),
                                     persist_directory='chroma_db_manual_openai')   # 설정이 다르면 폴더도 다르게
rag_chain = prompt | ChatOpenAI(model='gpt-5-nano') | StrOutputParser()
for q in ['환불 해줘', '돈 돌려줘', '핸드폰으로도 들을 수 있어?', '회사 주소가 어디예요?']:
    print('질문 :', q)
    print('  답 :', ask(q, chain=rag_chain, retriever=openai_store.as_retriever(search_kwargs={'k': 2})))

# 직접 입력하며 대화하려면
# while (q := input('Q&A (종료 q): ').strip()).lower() != 'q':     # 종료 확인을 먼저
#     print(ask(q, chain=rag_chain, retriever=openai_store.as_retriever(search_kwargs={'k': 2})), '\n')

In [23]:
# 체인 하나로 묶기 — 질문 문자열 하나만 넣으면 검색 → 증강 → 생성 (LCEL)
from langchain_core.runnables import RunnablePassthrough
full_chain = ({'context': retriever | format_docs, 'question': RunnablePassthrough()}
              | prompt | fake_llm | StrOutputParser())
print(full_chain.invoke('강의 자료는 어디서 받아요?'))

(모의 답변 · 컨텍스트 268자를 받음) 질문: 강의 자료나 소스 코드는 어디서 다운로드받나요?


- `{'context': retriever | format_docs, 'question': RunnablePassthrough()}` 는 같은 질문을 두 갈래로 보낸다 — 한쪽은 검색해서 글로, 한쪽은 그대로. 수업의 `while` 안에서 하던 '검색 → 이어 붙이기 → invoke' 를 체인 안으로 옮긴 것

## 정리

| 단계 | 코드 | 오늘 확인한 것 |
|---|---|---|
| ② 분할 | `chunk_size=160` | 110 → 25개(질문만 10 · 꼬리 조각 5) · 160 → 10개 = QA 10쌍 |
| ③ 임베딩 | `OpenAIEmbeddings('text-embedding-3-small')` 1,536차원 | 🧪 로컬 `ko-sroberta` 768차원 · 128 토큰 |
| ④ 저장 | `Chroma.from_documents(..., persist_directory=…)` | `chroma.sqlite3`(원문 · 메타데이터 · 기록) + UUID 폴더(HNSW 벡터 인덱스) |
| ④ 다시 열기 | `Chroma(persist_directory=…, embedding_function=…)` | 설정을 바꾸면 **폴더도 바꾼다** — 안 그러면 옛날 DB |
| ④ 검색 | `as_retriever(search_kwargs={'k': 2, 'filter': {...}})` | 항상 k개 · 필터는 먼저 거르고 순위 |
| FAISS | `FAISS.from_documents` · `save_local` · `load_local(..., allow_dangerous_deserialization=True)` | 같은 순위 · 저장은 직접 |
| ⑤ ⑥ ⑦ | `prompt \| llm \| StrOutputParser()` · `ask()` | `conttext` 오타 · 답 미출력 · 종료 확인 순서 고침 |